# Bronze Layer Notebook

### Purpose

This notebook ingests clinical trial data from the staging area into the Bronze layer, performs initial data validation, schema enforcement, and logging to ensure data quality and traceability.

---

### Steps Performed

1. **Import Required Libraries and Functions**
   - Loaded PySpark and utility functions for data processing and logging.

2. **Load Configuration**
   - Imported configuration settings for the notebook.

3. **Load Logging Functions**
   - Defined and loaded custom logging functions for tracking data operations.

4. **Read First Chunk of CSV Data**
   - Read the first staged CSV file from the S3 staging area.

5. **Parameterize Source Path**
   - Created and read a widget for dynamic source path selection.

6. **Read Data with Parameters**
   - Loaded the data using the parameterized source path and handled exceptions.

7. **Check Row Count**
   - Counted and logged the total number of records loaded.

8. **Check Existing Schema**
   - Printed and logged the schema of the loaded DataFrame.

9. **Verify Columns**
   - Checked and logged the column names in the DataFrame.

10. **Create Explicit Schema**
    - Defined an explicit schema for the clinical trial data.

11. **Verify Date Formats**
    - Displayed and logged the date columns to check their formats.

12. **Verify Schema**
    - Validated the schema after applying the explicit schema.

---

##### Importing the required libraries 

In [0]:
%run ./config

In [0]:
# Import required functions
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import input_file_name
from pyspark.sql.functions import lit

In [0]:
# Import logging dependencies directly
from datetime import datetime
from pyspark.sql import DataFrame

# Bronze logging functions
def log_bronze_read(df, source_path, operation="CSV Read"):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[BRONZE] [{operation}] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Source: {source_path} | Records: {count:,}")
    return count

def log_bronze_schema_validation(df, schema_applied=True):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[BRONZE] [Schema Validation] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Records: {count:,} | Schema Applied: {schema_applied}")
    return count

def log_bronze_audit_columns(df, file_name, version):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[BRONZE] [Audit Columns] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | File: {file_name} | Version: {version} | Records: {count:,}")
    return count

def log_bronze_write(df, table_name, write_mode="append"):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[BRONZE] [Write] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Table: {table_name} | Mode: {write_mode} | Records: {count:,}")
    return count

def log_null_check(df, column_name, operation_context=""):
    from pyspark.sql.functions import col
    total = df.count()
    null_count = df.filter(col(column_name).isNull()).count()
    print(f"[BRONZE] [Null Check] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Column: {column_name} | Context: {operation_context} | Total: {total:,} | Nulls: {null_count:,}")
    return {"total": total, "null_count": null_count}

def log_table_final_count(table_name, layer=""):
    try:
        count = spark.table(table_name).count()
        print(f"[{layer.upper()}] [Final Count] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Table: {table_name} | Records: {count:,}")
        return count
    except Exception as e:
        print(f"[{layer.upper()}] [Error] Could not get count for {table_name}: {str(e)}")
        return None

def log_gold_analytics(metric_name, metric_value, details=None):
    print(f"[GOLD] [Analytics] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Metric: {metric_name} | Value: {metric_value}")
    return metric_value

def log_gold_aggregation(df, aggregation_name, group_by_cols=None):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[GOLD] [Aggregation] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Name: {aggregation_name} | Records: {count:,}")
    return count

print("Logging functions loaded successfully")

##### Reading the First Chunk CSV file from the Staging

In [0]:
# Read the first staged file
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv("s3://clinical-trial4/staging/part1/")

# View data
display(df)

# Log CSV read operation
log_bronze_read(df, "s3://clinical-trial4/staging/part3/", "CSV Read")

In [0]:
%skip
# create widget
dbutils.widgets.text(
    "source_path",
    "s3://clinical-trial4/staging/part1/"
)

In [0]:
%skip
# read parameter
source_path = dbutils.widgets.get(
    "source_path"
)

print("Source Path :", source_path)

In [0]:
%skip
# read the data
try:

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(source_path)
    )

    print(
        "Source Count :",
        df.count()
    )

except Exception as e:

    print("Error :", e)

    raise e

##### Checking the row count

In [0]:
# Count total records
row_count = df.count()

print(f"Total Records Loaded : {row_count}")

# Log initial record count
log_bronze_read(row_count, "s3://clinical-trial4/staging/part3s/", "Initial Record Count")

##### Checking the Existing Schema

In [0]:
# Print schema
df.printSchema()

# Log schema inspection
log_bronze_schema_validation(df, schema_applied=False)

##### Verfying the columns

In [0]:
# checking the column names
print(df.columns)

# Log column inspection
log_bronze_read(df, "DataFrame", "Column Names Verification")

##### creating Explicit schemas

In [0]:
# Importing the libraries
from pyspark.sql.types import *

schema = StructType([

    StructField("NCT Number", StringType(), True),
    StructField("Study Title", StringType(), True),
    StructField("Acronym", StringType(), True),
    StructField("Study Status", StringType(), True),
    StructField("Brief Summary", StringType(), True),
    StructField("Study Results", StringType(), True),
    StructField("Conditions", StringType(), True),
    StructField("Interventions", StringType(), True),
    StructField("Primary Outcome Measures", StringType(), True),
    StructField("Secondary Outcome Measures", StringType(), True),
    StructField("Other Outcome Measures", StringType(), True),
    StructField("Sponsor", StringType(), True),
    StructField("Collaborators", StringType(), True),
    StructField("Sex", StringType(), True),

    # Age values are typically Adult, Child, Older Adult etc.
    StructField("Age", StringType(), True),

    StructField("Phases", StringType(), True),

    # Numeric column
    StructField("Enrollment", IntegerType(), True),

    StructField("Funder Type", StringType(), True),
    StructField("Study Type", StringType(), True),
    StructField("Study Design", StringType(), True),
    StructField("Other IDs", StringType(), True),

    # Date columns
    StructField("Start Date", DateType(), True),
    StructField("Primary Completion Date", DateType(), True),
    StructField("Completion Date", DateType(), True),
    StructField("First Posted", DateType(), True),
    StructField("Results First Posted", DateType(), True),
    StructField("Last Update Posted", DateType(), True),

    StructField("Locations", StringType(), True)

])

# Log schema creation
print("[BRONZE] [Schema Created] Explicit schema defined with 28 fields")

##### verfying the date format

In [0]:
display(
    df.select(
        "Start Date",
        "Primary Completion Date",
        "Completion Date",
        "First Posted",
        "Results First Posted",
        "Last Update Posted"
    )
)

# Log date format verification
log_bronze_read(df, "Date Columns", "Date Format Verification")

##### Verfying the schema

In [0]:
# Reading the schemas
df = spark.read.csv(
    "s3://clinical-trial4/staging/part1/",
    header=True,
    schema=schema
)

# Log schema application
log_bronze_read(df, "s3://clinical-trial4/staging/part1/", "CSV Read with Schema")

In [0]:
df.printSchema()

# Log schema verification
log_bronze_schema_validation(df, schema_applied=True)

In [0]:
# Checking the datatypes
for field in df.schema.fields:
    print(f"{field.name} --> {field.dataType}")

# Log datatype verification
log_bronze_schema_validation(df, schema_applied=True)

In [0]:
# -- Data Validation -- #
# verifying the row count
row_count_validation = df.count()
print("Row Count:", row_count_validation)

# Log validation count
log_bronze_read(row_count_validation, "DataFrame", "Data Validation Count")

In [0]:
# Viewing the sample data 
display(df.limit(10))

# Log sample display
log_bronze_read(10, "DataFrame", "Sample Data Display")

In [0]:
# checking the null values for important columns
from pyspark.sql.functions import col, count, when

display(
df.select(
    count(when(col("NCT Number").isNull(), True)).alias("Null_NCT_Number"),
    count(when(col("Study Title").isNull(), True)).alias("Null_Study_Title")
))

# Log null check validation
log_null_check(df, "NCT Number", "Bronze Layer - Key Column Validation")

##### Adding the audit columns

In [0]:
# File information
file_name = "part1"
version = 0

from pyspark.sql.functions import current_timestamp, lit

df_bronze = df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file_name", lit(file_name)) \
    .withColumn("version", lit(version))

# Log audit columns addition
log_bronze_audit_columns(df_bronze, file_name, version)

In [0]:
display(
    df_bronze.select(
        "source_file_name",
        "version",
        "ingestion_timestamp"
    )
)

# Log audit column verification
log_bronze_audit_columns(df_bronze, file_name, version)

##### Writing the bronze data as delta in S3

In [0]:
# Get bronze path from config
bronze_path = cfg['bronze_path']

# Log path configuration
print(f"[BRONZE] [Path Configured] {bronze_path}")

In [0]:
# writing the data as delta records in bronze prefix
df_bronze.write.format("delta") \
.mode("append") \
.option("delta.columnMapping.mode", "name") \
.save(bronze_path)

# Log bronze write operation
log_bronze_write(df_bronze, "clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze", "append")

In [0]:
# verifying the write 
bronze_df = spark.read \
    .format("delta") \
    .load(bronze_path)

display(bronze_df.limit(5))

# Log read verification
log_bronze_read(bronze_df, bronze_path, "Bronze Data Read Verification")

In [0]:
# checking the count 
print("Bronze Row Count:", bronze_df.count())

# Log final table verification
log_table_final_count("clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze", "bronze")

#### Catalog Creation

In [0]:
# Create catalog from config
spark.sql(f"CREATE CATALOG IF NOT EXISTS {cfg['catalog']}")

# Log catalog creation
print(f"[BRONZE] [Catalog Created] {cfg['catalog']}")

##### Creating the bronze schema


In [0]:
# Create bronze schema from config
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {cfg['catalog']}.{cfg['schema_bronze']}")

# Log schema creation
print(f"[BRONZE] [Schema Created] {cfg['catalog']}.{cfg['schema_bronze']}")

##### Creating the bronze table

In [0]:
# Create bronze table from config
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cfg['bronze_table']}
USING DELTA
LOCATION '{cfg['bronze_path']}'
""")

# Log table creation
print(f"[BRONZE] [Table Created] {cfg['bronze_table']}")

##### Verifying the table

In [0]:
# Verify table count from config
verify_count = spark.sql(f"""
SELECT COUNT(*) as count
FROM {cfg['bronze_table']}
""").collect()[0][0]

print(f"Table Count: {verify_count}")

# Log table verification
log_table_final_count(cfg['bronze_table'], "bronze")

#### Bronze layer validation - EDA (Exploratory data analysis)

In [0]:
# Bronze layer - Record count
total_records_df = spark.sql("""
SELECT COUNT(*) AS total_records
FROM clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze
""")

display(total_records_df)

# Log analytics
log_gold_analytics("Record Count", "Bronze layer total records")

In [0]:
# Check null values in key columns
null_check_df = spark.sql("""
SELECT
  COUNT(*) AS total_records,
  COUNT(`NCT Number`) AS nct_number_count
FROM clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze
""")

display(null_check_df)

# Log null check analytics
log_gold_analytics("Null Check", "Key column null value verification")

In [0]:
# Study status distribution (excluding nulls)
status_dist_df = spark.sql("""
SELECT
  `Study Status`,
  COUNT(*) AS study_count
FROM clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze
WHERE `Study Status` IS NOT NULL
GROUP BY `Study Status`
ORDER BY study_count DESC
LIMIT 10
""")

display(status_dist_df)

# Log aggregation analytics
log_gold_aggregation(status_dist_df, "Study Status Distribution", ["Study Status"])

In [0]:
# Study type distribution (excluding nulls)
type_dist_df = spark.sql("""
SELECT
  `Study Type`,
  COUNT(*) AS study_count
FROM clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze
WHERE `Study Type` IS NOT NULL
GROUP BY `Study Type`
ORDER BY study_count DESC
LIMIT 10
""")

display(type_dist_df)

# Log aggregation analytics
log_gold_aggregation(type_dist_df, "Study Type Distribution", ["Study Type"])

In [0]:
# Check date range
date_range_df = spark.sql("""
SELECT
  MIN(`Start Date`) AS min_start_date,
  MAX(`Start Date`) AS max_start_date
FROM clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze
""")

display(date_range_df)

# Log date range analytics
log_gold_analytics("Date Range Check", "Start date min/max verification")

In [0]:
# Describe table history
history_df = spark.sql("""
DESCRIBE HISTORY clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze
""")

display(history_df)

# Log history analytics
log_gold_analytics("Table History", "Delta table history verification")